# 14 - Trajectory Generation

## What / Why / How

**What we are trying to do:** Generate smooth robot motion profiles instead of jumping directly from start to goal.

**Why this matters:** Robots need trajectories that respect velocity, acceleration, smoothness, and hardware limits.

**How we will do it:** Compare cubic, quintic, and trapezoidal profiles, then inspect endpoint behavior and duration.

## Goal

Robots need trajectories, not just goals.

You will implement:

- Cubic time scaling
- Quintic time scaling
- Trapezoidal velocity profiles

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
def cubic_s(t):
    return 3*t**2 - 2*t**3

def quintic_s(t):
    return 10*t**3 - 15*t**4 + 6*t**5

time = np.linspace(0, 1, 101)
q0, q1 = -1.0, 2.0
cubic_q = q0 + (q1 - q0) * cubic_s(time)
quintic_q = q0 + (q1 - q0) * quintic_s(time)
cubic_v = np.gradient(cubic_q, time)
quintic_v = np.gradient(quintic_q, time)
print("cubic endpoint velocities:", cubic_v[0], cubic_v[-1])
print("quintic endpoint velocities:", quintic_v[0], quintic_v[-1])

In [ ]:
def trapezoid_profile(distance, vmax, amax, dt=0.02):
    t_acc = vmax / amax
    d_acc = 0.5 * amax * t_acc**2
    if 2 * d_acc > distance:
        t_acc = np.sqrt(distance / amax)
        t_flat = 0.0
        vmax = amax * t_acc
    else:
        t_flat = (distance - 2*d_acc) / vmax
    total = 2*t_acc + t_flat
    ts = np.arange(0, total + dt, dt)
    xs, vs = [], []
    for t in ts:
        if t < t_acc:
            v = amax * t
            x = 0.5 * amax * t**2
        elif t < t_acc + t_flat:
            v = vmax
            x = 0.5 * amax * t_acc**2 + vmax * (t - t_acc)
        else:
            tau = t - t_acc - t_flat
            v = vmax - amax * tau
            x = distance - 0.5 * amax * max(total - t, 0)**2
        xs.append(x)
        vs.append(max(v, 0))
    return ts, np.array(xs), np.array(vs)

ts, xs, vs = trapezoid_profile(distance=3.0, vmax=1.2, amax=1.5)
print("duration:", round(float(ts[-1]), 3), "final distance:", round(float(xs[-1]), 3))

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(8, 3))
    plt.plot(time, cubic_q, label="cubic q")
    plt.plot(time, quintic_q, label="quintic q")
    plt.plot(ts / ts[-1], xs / xs[-1] * (q1 - q0) + q0, label="trapezoid scaled")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    plot_unavailable()

## Exercises

1. Add acceleration plots.
2. Make a trajectory through three waypoints.
3. Explain why smooth acceleration protects hardware.